# Trader Behavior vs Market Sentiment Analysis

## Objective
Analyze how Bitcoin market sentiment (Fear & Greed Index) influences
trader behavior and performance using historical trading data from Hyperliquid.

## Key Questions
- Does trader profitability change across Fear and Greed regimes?
- How does risk behavior (trade size, losses) vary with sentiment?
- Are there identifiable patterns among consistently profitable traders?

## Datasets
- Fear & Greed Index (daily sentiment classification)
- Hyperliquid historical trader data (execution-level trades)


In [1]:
import pandas as pd

fear_path = "../dataset/fear_greed_index.csv"
trades_path = "../dataset/historical_data.csv"

fear_df = pd.read_csv(fear_path)
trades_df = pd.read_csv(trades_path)

print(fear_df.shape)
print(trades_df.shape)

(2644, 4)
(211224, 16)


In [2]:
import pandas as pd

fear_df = pd.read_csv("../dataset/fear_greed_index.csv")
trades_df = pd.read_csv("../dataset/historical_data.csv")

In [3]:
fear_df.head()
fear_df.info()
fear_df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2644 entries, 0 to 2643
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   timestamp       2644 non-null   int64 
 1   value           2644 non-null   int64 
 2   classification  2644 non-null   object
 3   date            2644 non-null   object
dtypes: int64(2), object(2)
memory usage: 82.8+ KB


,timestamp,value,classification,date
count,2.644000e+03,2644.000000,2644,2644
unique,NaN,NaN,5,2644
top,NaN,NaN,Fear,2025-05-02
freq,NaN,NaN,781,1
mean,1.631899e+09,46.981089,NaN,NaN
std,6.597967e+07,21.827680,NaN,NaN
min,1.517463e+09,5.000000,NaN,NaN
25%,1.574811e+09,28.000000,NaN,NaN
50%,1.631900e+09,46.000000,NaN,NaN
75%,1.688989e+09,66.000000,NaN,NaN


In [4]:
fear_df['classification'].value_counts()

classification
Fear             781
Greed            633
Extreme Fear     508
Neutral          396
Extreme Greed    326
Name: count, dtype: int64

In [5]:
trades_df.shape
trades_df.columns

Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp'],
      dtype='object')

In [6]:
trades_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211224 entries, 0 to 211223
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Account           211224 non-null  object 
 1   Coin              211224 non-null  object 
 2   Execution Price   211224 non-null  float64
 3   Size Tokens       211224 non-null  float64
 4   Size USD          211224 non-null  float64
 5   Side              211224 non-null  object 
 6   Timestamp IST     211224 non-null  object 
 7   Start Position    211224 non-null  float64
 8   Direction         211224 non-null  object 
 9   Closed PnL        211224 non-null  float64
 10  Transaction Hash  211224 non-null  object 
 11  Order ID          211224 non-null  int64  
 12  Crossed           211224 non-null  bool   
 13  Fee               211224 non-null  float64
 14  Trade ID          211224 non-null  float64
 15  Timestamp         211224 non-null  float64
dtypes: bool(1), float64(

### Trader Performance Metrics
- Closed PnL
- Win/Loss indicator
- Absolute PnL

### Risk & Behavior Metrics
- Size USD
- Execution Price
- Fee

### Identity & Grouping
- Account
- Coin

### Time Alignment
- Timestamp IST (used to derive trade date)

### Columns Excluded from Analysis
- Transaction Hash (identifier only)
- Order ID (not required for behavior analysis)
- Trade ID
- Crossed flag
- Start Position

### Assumptions
- Each row represents a single execution contributing to realized PnL.
- Daily market sentiment applies uniformly to all trades executed on that day.
- Closed PnL accurately reflects trade outcome and is suitable for performance evaluation.

In [7]:
# ---- Fear–Greed data cleaning ----

fear = fear_df.copy()

# Convert date column to datetime
fear['date'] = pd.to_datetime(fear['date'], errors='coerce')

# Keep only required columns
fear = fear[['date', 'value', 'classification']]

# Rename for clarity
fear.rename(columns={
    'value': 'sentiment_score',
    'classification': 'sentiment'
}, inplace=True)

fear.head()

,date,sentiment_score,sentiment
0,2018-02-01,30,Fear
1,2018-02-02,15,Extreme Fear
2,2018-02-03,40,Fear
3,2018-02-04,24,Extreme Fear
4,2018-02-05,11,Extreme Fear


In [8]:
fear.info()
fear.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2644 entries, 0 to 2643
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date             2644 non-null   datetime64[ns]
 1   sentiment_score  2644 non-null   int64         
 2   sentiment        2644 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 62.1+ KB


date               0
sentiment_score    0
sentiment          0
dtype: int64

In [9]:
trades = trades_df.copy()

In [10]:
# Convert Timestamp IST to datetime
trades['Timestamp IST'] = pd.to_datetime(
    trades['Timestamp IST'], errors='coerce'
)

# Extract date for merging
trades['date'] = trades['Timestamp IST'].dt.date
trades['date'] = pd.to_datetime(trades['date'])

trades[['Timestamp IST', 'date']].head()

,Timestamp IST,date
0,2024-02-12 22:50:00,2024-02-12
1,2024-02-12 22:50:00,2024-02-12
2,2024-02-12 22:50:00,2024-02-12
3,2024-02-12 22:50:00,2024-02-12
4,2024-02-12 22:50:00,2024-02-12


In [11]:
numeric_cols = [
    'Closed PnL',
    'Size USD',
    'Execution Price',
    'Fee'
]

for col in numeric_cols:
    trades[col] = pd.to_numeric(trades[col], errors='coerce')

In [12]:
# Drop trades without PnL or size
trades = trades.dropna(subset=['Closed PnL', 'Size USD'])

trades.shape

(211224, 17)

In [13]:
# Profitability flag
trades['is_profitable'] = trades['Closed PnL'] > 0

# Absolute PnL (loss magnitude analysis)
trades['abs_pnl'] = trades['Closed PnL'].abs()

In [14]:
trades.info()
trades.isna().sum().sort_values(ascending=False).head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211224 entries, 0 to 211223
Data columns (total 19 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Account           211224 non-null  object        
 1   Coin              211224 non-null  object        
 2   Execution Price   211224 non-null  float64       
 3   Size Tokens       211224 non-null  float64       
 4   Size USD          211224 non-null  float64       
 5   Side              211224 non-null  object        
 6   Timestamp IST     79225 non-null   datetime64[ns]
 7   Start Position    211224 non-null  float64       
 8   Direction         211224 non-null  object        
 9   Closed PnL        211224 non-null  float64       
 10  Transaction Hash  211224 non-null  object        
 11  Order ID          211224 non-null  int64         
 12  Crossed           211224 non-null  bool          
 13  Fee               211224 non-null  float64       
 14  Trad

Timestamp IST      131999
date               131999
Account                 0
Execution Price         0
Coin                    0
dtype: int64

In [15]:
fear.to_csv("../dataset/fear_greed_cleaned.csv", index=False)
trades.to_csv("../dataset/trader_data_cleaned.csv", index=False)

In [16]:
import pandas as pd

fear = pd.read_csv("../dataset/fear_greed_cleaned.csv", parse_dates=['date'])
trades = pd.read_csv("../dataset/trader_data_cleaned.csv", parse_dates=['date'])

fear.shape, trades.shape

((2644, 3), (211224, 19))

In [17]:
print("Fear-Greed date range:", fear['date'].min(), "to", fear['date'].max())
print("Trades date range:", trades['date'].min(), "to", trades['date'].max())

Fear-Greed date range: 2018-02-01 00:00:00 to 2025-05-02 00:00:00
Trades date range: 2023-01-05 00:00:00 to 2025-12-04 00:00:00


- Only overlapping dates will be included to ensure valid sentiment labeling.

In [18]:
merged_df = trades.merge(
    fear,
    on='date',
    how='inner'
)

merged_df.shape

(35864, 21)

In [19]:
merged_df['sentiment'].value_counts()

sentiment
Fear             13869
Greed            11292
Extreme Greed     5621
Neutral           2756
Extreme Fear      2326
Name: count, dtype: int64

In [20]:
merged_df[['sentiment', 'sentiment_score']].isna().sum()

sentiment          0
sentiment_score    0
dtype: int64

In [21]:
print("Trades before merge:", trades.shape[0])
print("Trades after merge:", merged_df.shape[0])

Trades before merge: 211224
Trades after merge: 35864


In [22]:
merged_df.to_csv(
    "../dataset/merged_trader_sentiment.csv",
    index=False
)

- Merged trader executions with daily Bitcoin Fear & Greed Index using trade date
- Inner join ensures all trades have valid sentiment labels
- Final dataset represents sentiment-aligned trade behavior
- Dataset is now ready for exploratory and behavioral analysis